In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [2]:
# ---------- 1. 准备数据 ----------
# 把图片（0-255像素）转为张量（0-1），并标准化（让数据分布更稳定）
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# 下载训练集和测试集（第一次运行会下载，耐心等几十秒）
train_set = torchvision.datasets.MNIST(root='./data', train=True, transform=transform,download=True)
test_set = torchvision.datasets.MNIST(root='./data', train=False, transform=transform,download=True)

# DataLoader：按批次(Batch)喂数据，每批64张，并打乱顺序
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

In [3]:
# ---------- 2. 定义模型 ----------
class Net(nn.Module):
    def __init__(self):
        super().__init__()

        # 卷积层
        # Conv2d（输入通道，输出通道，卷积核大小）
        self.conv1 = nn.Conv2d(1, 6, 5) # 输入1通道，输出6通道，卷积核5x5
        self.conv2 = nn.Conv2d(6, 16, 5) # 输入6通道，输出16通道，卷积核5x5

        # 全连接层
        # Linear(输入维度，输出维度)
        self.fc1 = nn.Linear(16 * 4 * 4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10) # 输出10类

    def forward(self, x):
        # 卷积1 -> Relu -> 池化
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, (2,2))

        # 卷积2 -> Relu -> 池化
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)

        # 展平多维的卷积图成一维的向量
        x = torch.flatten(x, 1)

        # 全连接层 + Relu
        x= F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        # 输出层(10类)
        x = self.fc3(x)
        # print(x.shape)
        return x

net = Net()
print(net)

Net(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=256, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


In [4]:
# ---------- 3. 定义损失函数和优化器 ----------
criterion = nn.CrossEntropyLoss() # 交叉熵损失：专用于分类
optimizer = optim.SGD(net.parameters(), lr=0.04) # 随机梯度下降，学习率0.04

In [6]:
# ---------- 4. 训练循环（铁打的5步舞曲） ----------
num_epochs = 5 # 整个数据集跑5遍

for epoch in range(num_epochs):
    running_loss = 0.0
    for images, labels in train_loader:
        # print(images.shape, labels.shape)
        # ① 梯度清零（重要！）
        optimizer.zero_grad()

        # ② 前向传播：算出预测值
        outputs = net(images)

        # ③ 计算损失
        loss = criterion(outputs, labels)

         # ④ 反向传播：自动求梯度
        loss.backward()
        # print(net.fc1.weight.grad)

        # print("=====step()之前conv1偏置=====")
        # print(net.conv1.bias)
        # ⑤ 更新权重
        optimizer.step()
        # print("=====step()之后conv1偏置=====")
        # print(net.conv1.bias)
        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")
    print("Finished Training")

Epoch [1/5], Loss: 0.0535
Finished Training
Epoch [2/5], Loss: 0.0452
Finished Training
Epoch [3/5], Loss: 0.0392
Finished Training
Epoch [4/5], Loss: 0.0349
Finished Training
Epoch [5/5], Loss: 0.0313
Finished Training


In [7]:
# ---------- 5. 简单测试（看准确率） ---------
correct = 0
total = 0
with torch.no_grad(): # 测试时不记录梯度，省内存
    for images, labels in test_loader:
        outputs = net(images)
        _, predicted = torch.max(outputs, 1) # 取每行最大值的索引作为预测类别
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f"Accuracy on test set: {100 * correct / total:.2f}%")

Accuracy on test set: 98.80%
